# Feature Selection on Wine Quality Dataset
## A Complete Machine Learning Practice Notebook

**Objective:** Master feature selection using Filter, Wrapper, and Embedded methods

**Dataset:** Wine Quality (White) - winequality-white.csv

**Target Variable:** Quality (regression task)

This notebook provides:
- Complete data exploration and preprocessing
- Three major feature selection approaches
- Comparative analysis of methods
- Professional visualizations
- Detailed explanations and best practices

## Cell 1: Import All Required Libraries

In [ ]:
# Import data manipulation libraries
import pandas as pd
import numpy as np

# Import visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Import machine learning libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Lasso, LassoCV
from sklearn.feature_selection import (
    VarianceThreshold,
    mutual_info_regression,
    f_regression,
    SequentialFeatureSelector,
    RFE
)
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Configure matplotlib for better visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All libraries imported successfully!")

## Cell 2: Upload Dataset to Google Colab

**Instructions:**
1. Click on the folder icon in the left sidebar
2. Click "Upload to session storage"
3. Select the winequality-white.csv file
4. Run the cell to load the data

In [ ]:
# For Google Colab: Upload file
from google.colab import files
print("Please upload the winequality-white.csv file:")
uploaded = files.upload()

# Load the dataset
df = pd.read_csv('winequality-white.csv', sep=';')
print("✓ Dataset uploaded successfully!")

## Cell 3: Data Loading and Exploration

In this cell, we:
1. Display dataset structure and statistics
2. Check for missing values
3. Separate features (X) and target variable (y)

In [ ]:
# Display first few rows
print("="*80)
print("FIRST 5 ROWS OF THE DATASET")
print("="*80)
print(df.head())

# Display dataset shape
print("\n" + "="*80)
print("DATASET SHAPE")
print("="*80)
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

# Display data types
print("\n" + "="*80)
print("DATA TYPES")
print("="*80)
print(df.dtypes)

# Display summary statistics
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)
print(df.describe())

# Check for missing values
print("\n" + "="*80)
print("MISSING VALUES CHECK")
print("="*80)
missing_values = df.isnull().sum()
print(f"Total missing values: {missing_values.sum()}")
if missing_values.sum() == 0:
    print("✓ No missing values found!")

# Separate features and target variable
X = df.drop('quality', axis=1)
y = df['quality']

print("\n" + "="*80)
print("FEATURE NAMES (X)")
print("="*80)
feature_names = X.columns.tolist()
for i, feature in enumerate(feature_names, 1):
    print(f"{i:2d}. {feature}")

print(f"\nTotal Features: {len(feature_names)}")
print(f"Target Variable: 'quality' (shape: {y.shape})")

## Cell 4: Correlation Analysis

In this cell, we:
1. Compute Pearson correlation matrix
2. Visualize using heatmap
3. Identify and remove highly correlated features

In [ ]:
# Compute correlation matrix
correlation_matrix = df.corr()

# Visualize correlation matrix
plt.figure(figsize=(14, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix - Wine Quality Dataset', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Identify highly correlated pairs
print("\n" + "="*80)
print("HIGHLY CORRELATED FEATURE PAIRS (|correlation| > 0.90)")
print("="*80)

high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > 0.90:
            feature1 = correlation_matrix.columns[i]
            feature2 = correlation_matrix.columns[j]
            corr_value = correlation_matrix.iloc[i, j]
            high_corr_pairs.append((feature1, feature2, corr_value))
            print(f"{feature1:25s} <-> {feature2:25s}: {corr_value:7.4f}")

if not high_corr_pairs:
    print("No highly correlated pairs found (|r| > 0.90)")

print(f"\nOriginal feature count: {len(feature_names)}")

## Cell 5: Baseline Model (All Features)

In this cell, we:
1. Split data into training (80%) and testing (20%)
2. Standardize features
3. Train Linear Regression model with all features
4. Evaluate and store baseline results

In [ ]:
print("="*80)
print("BASELINE MODEL: TRAINING WITH ALL FEATURES")
print("="*80)

# Train-Test Split
X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train_base.shape[0]}")
print(f"Testing set size: {X_test_base.shape[0]}")

# Standardize features
scaler_base = StandardScaler()
X_train_base_scaled = scaler_base.fit_transform(X_train_base)
X_test_base_scaled = scaler_base.transform(X_test_base)

# Train baseline model
baseline_model = LinearRegression()
baseline_model.fit(X_train_base_scaled, y_train_base)

# Make predictions
y_pred_base = baseline_model.predict(X_test_base_scaled)

# Calculate metrics
baseline_rmse = np.sqrt(mean_squared_error(y_test_base, y_pred_base))
baseline_mae = mean_absolute_error(y_test_base, y_pred_base)
baseline_r2 = r2_score(y_test_base, y_pred_base)

print("\n" + "-"*80)
print("BASELINE MODEL PERFORMANCE")
print("-"*80)
print(f"Number of Features: {X_train_base.shape[1]}")
print(f"RMSE (Root Mean Squared Error): {baseline_rmse:.4f}")
print(f"MAE  (Mean Absolute Error):     {baseline_mae:.4f}")
print(f"R²   (R-squared Score):         {baseline_r2:.4f}")

# Store baseline results
results = {
    'Method': ['Baseline (All Features)'],
    'Number_of_Features': [X_train_base.shape[1]],
    'RMSE': [baseline_rmse],
    'MAE': [baseline_mae],
    'R2': [baseline_r2]
}

## Cell 6: Filter Method 1 - Variance Threshold

In [ ]:
print("\n" + "="*80)
print("FILTER METHOD 1: VARIANCE THRESHOLD")
print("="*80)

# Apply Variance Threshold
variance_threshold = VarianceThreshold(threshold=0.01)
X_var_filtered = variance_threshold.fit_transform(X_train_base)

# Get selected and removed features
selected_indices_var = variance_threshold.get_support(indices=True)
selected_features_var = X.columns[selected_indices_var].tolist()
removed_features_var = X.columns[~variance_threshold.get_support()].tolist()

print(f"\nOriginal number of features: {X_train_base.shape[1]}")
print(f"Features removed: {len(removed_features_var)}")
print(f"Features remaining: {len(selected_features_var)}")

print("\n" + "-"*80)
print("SELECTED FEATURES (High Variance)")
print("-"*80)
for i, feat in enumerate(selected_features_var, 1):
    print(f"{i:2d}. {feat}")

## Cell 7: Filter Method 2 - Mutual Information

In [ ]:
print("\n" + "="*80)
print("FILTER METHOD 2: MUTUAL INFORMATION")
print("="*80)

# Calculate Mutual Information scores
mi_scores = mutual_info_regression(X_train_base_scaled, y_train_base, random_state=42)

# Create dataframe for visualization
mi_df = pd.DataFrame({
    'Feature': X.columns,
    'MI_Score': mi_scores
}).sort_values('MI_Score', ascending=False)

print("\n" + "-"*80)
print("MUTUAL INFORMATION SCORES (All Features)")
print("-"*80)
print(mi_df.to_string(index=False))

# Plot Mutual Information
plt.figure(figsize=(10, 6))
plt.barh(mi_df['Feature'], mi_df['MI_Score'], color='steelblue')
plt.xlabel('Mutual Information Score', fontsize=12, fontweight='bold')
plt.ylabel('Features', fontsize=12, fontweight='bold')
plt.title('Mutual Information Scores - All Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Select top 5 features
top_5_mi = mi_df.head(5)['Feature'].tolist()

print("\n" + "-"*80)
print("TOP 5 FEATURES (Mutual Information)")
print("-"*80)
for i, feat in enumerate(top_5_mi, 1):
    mi_score = mi_df[mi_df['Feature'] == feat]['MI_Score'].values[0]
    print(f"{i}. {feat:30s} (MI Score: {mi_score:.4f})")

## Cell 8: Filter Method 3 - ANOVA F-Test

In [ ]:
print("\n" + "="*80)
print("FILTER METHOD 3: ANOVA F-TEST")
print("="*80)

# Calculate ANOVA F-scores
f_scores, p_values = f_regression(X_train_base_scaled, y_train_base)

# Create dataframe
f_df = pd.DataFrame({
    'Feature': X.columns,
    'F_Score': f_scores,
    'P_Value': p_values
}).sort_values('F_Score', ascending=False)

print("\n" + "-"*80)
print("ANOVA F-TEST SCORES (All Features)")
print("-"*80)
print(f_df.to_string(index=False))

# Plot ANOVA F-scores
plt.figure(figsize=(10, 6))
plt.barh(f_df['Feature'], f_df['F_Score'], color='coral')
plt.xlabel('ANOVA F-Score', fontsize=12, fontweight='bold')
plt.ylabel('Features', fontsize=12, fontweight='bold')
plt.title('ANOVA F-Test Scores - All Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Select top 5 features
top_5_f = f_df.head(5)['Feature'].tolist()

print("\n" + "-"*80)
print("TOP 5 FEATURES (ANOVA F-Test)")
print("-"*80)
for i, feat in enumerate(top_5_f, 1):
    f_score = f_df[f_df['Feature'] == feat]['F_Score'].values[0]
    print(f"{i}. {feat:30s} (F-Score: {f_score:.4f})")

## Cell 9: Filter Method 4 - Correlation-Based Selection

In [ ]:
print("\n" + "="*80)
print("FILTER METHOD 4: CORRELATION-BASED SELECTION")
print("="*80)

# Calculate correlations with target
correlations = X.corrwith(y).abs().sort_values(ascending=False)
corr_df = pd.DataFrame({
    'Feature': correlations.index,
    'Correlation': correlations.values
})

print("\n" + "-"*80)
print("FEATURE-TARGET CORRELATIONS")
print("-"*80)
print(corr_df.to_string(index=False))

# Plot correlations
plt.figure(figsize=(10, 6))
plt.barh(corr_df['Feature'], corr_df['Correlation'], color='mediumseagreen')
plt.xlabel('Absolute Correlation with Target', fontsize=12, fontweight='bold')
plt.ylabel('Features', fontsize=12, fontweight='bold')
plt.title('Feature-Target Correlations', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Select top 5 features
top_5_corr = corr_df.head(5)['Feature'].tolist()

print("\n" + "-"*80)
print("TOP 5 FEATURES (Correlation-Based)")
print("-"*80)
for i, feat in enumerate(top_5_corr, 1):
    corr_val = corr_df[corr_df['Feature'] == feat]['Correlation'].values[0]
    print(f"{i}. {feat:30s} (Correlation: {corr_val:.4f})")

## Cell 10: Filter Method Evaluation

In [ ]:
print("\n" + "="*80)
print("FILTER METHOD EVALUATION")
print("="*80)

# Combine all filter-selected features
filter_features = list(set(top_5_mi + top_5_f + top_5_corr + selected_features_var))
filter_features.sort()

print(f"\nTotal unique features selected: {len(filter_features)}")
print("\nSelected Features:")
for i, feat in enumerate(filter_features, 1):
    print(f"{i:2d}. {feat}")

# Prepare data
X_train_filter = X_train_base[filter_features]
X_test_filter = X_test_base[filter_features]

# Standardize
scaler_filter = StandardScaler()
X_train_filter_scaled = scaler_filter.fit_transform(X_train_filter)
X_test_filter_scaled = scaler_filter.transform(X_test_filter)

# Train and evaluate
filter_model = LinearRegression()
filter_model.fit(X_train_filter_scaled, y_train_base)
y_pred_filter = filter_model.predict(X_test_filter_scaled)

filter_rmse = np.sqrt(mean_squared_error(y_test_base, y_pred_filter))
filter_mae = mean_absolute_error(y_test_base, y_pred_filter)
filter_r2 = r2_score(y_test_base, y_pred_filter)

print("\n" + "-"*80)
print("FILTER METHOD MODEL PERFORMANCE")
print("-"*80)
print(f"Number of Features: {len(filter_features)}")
print(f"RMSE: {filter_rmse:.4f}")
print(f"MAE:  {filter_mae:.4f}")
print(f"R²:   {filter_r2:.4f}")

# Store results
results['Method'].append('Filter Methods')
results['Number_of_Features'].append(len(filter_features))
results['RMSE'].append(filter_rmse)
results['MAE'].append(filter_mae)
results['R2'].append(filter_r2)

## Cell 11: Wrapper Method 1 - Forward Selection

In [ ]:
print("\n" + "="*80)
print("WRAPPER METHOD 1: FORWARD SELECTION")
print("="*80)

# Forward Selection
forward_selector = SequentialFeatureSelector(
    LinearRegression(),
    n_features_to_select=5,
    direction='forward',
    n_jobs=-1
)

forward_selector.fit(X_train_base_scaled, y_train_base)

# Get selected features
forward_indices = forward_selector.get_support(indices=True)
forward_features = X.columns[forward_indices].tolist()

print(f"\nSelected {len(forward_features)} features:")
for i, feat in enumerate(forward_features, 1):
    print(f"{i}. {feat}")

## Cell 12: Wrapper Method 2 - Backward Elimination

In [ ]:
print("\n" + "="*80)
print("WRAPPER METHOD 2: BACKWARD ELIMINATION")
print("="*80)

# Backward Elimination
backward_selector = SequentialFeatureSelector(
    LinearRegression(),
    n_features_to_select=5,
    direction='backward',
    n_jobs=-1
)

backward_selector.fit(X_train_base_scaled, y_train_base)

# Get selected features
backward_indices = backward_selector.get_support(indices=True)
backward_features = X.columns[backward_indices].tolist()

print(f"\nSelected {len(backward_features)} features:")
for i, feat in enumerate(backward_features, 1):
    print(f"{i}. {feat}")

## Cell 13: Wrapper Method 3 - RFE (Recursive Feature Elimination)

In [ ]:
print("\n" + "="*80)
print("WRAPPER METHOD 3: RECURSIVE FEATURE ELIMINATION (RFE)")
print("="*80)

# RFE
rfe = RFE(
    estimator=LinearRegression(),
    n_features_to_select=5,
    step=1
)

rfe.fit(X_train_base_scaled, y_train_base)

# Get selected features and rankings
rfe_indices = rfe.get_support(indices=True)
rfe_features = X.columns[rfe_indices].tolist()

rfe_ranking = pd.DataFrame({
    'Feature': X.columns,
    'Ranking': rfe.ranking_
}).sort_values('Ranking')

print(f"\nSelected {len(rfe_features)} features:")
for i, feat in enumerate(rfe_features, 1):
    print(f"{i}. {feat}")

print("\n" + "-"*80)
print("RFE FEATURE RANKINGS")
print("-"*80)
print(rfe_ranking.to_string(index=False))

# Plot RFE rankings
plt.figure(figsize=(10, 6))
plt.barh(rfe_ranking['Feature'], rfe_ranking['Ranking'], color='mediumpurple')
plt.xlabel('Feature Ranking (Lower is Better)', fontsize=12, fontweight='bold')
plt.ylabel('Features', fontsize=12, fontweight='bold')
plt.title('RFE Feature Rankings', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Cell 14: Wrapper Method Evaluation

In [ ]:
print("\n" + "="*80)
print("WRAPPER METHOD EVALUATION")
print("="*80)

print(f"\nUsing features selected by RFE")
print(f"Number of Features: {len(rfe_features)}")

# Prepare data
X_train_wrapper = X_train_base[rfe_features]
X_test_wrapper = X_test_base[rfe_features]

# Standardize
scaler_wrapper = StandardScaler()
X_train_wrapper_scaled = scaler_wrapper.fit_transform(X_train_wrapper)
X_test_wrapper_scaled = scaler_wrapper.transform(X_test_wrapper)

# Train and evaluate
wrapper_model = LinearRegression()
wrapper_model.fit(X_train_wrapper_scaled, y_train_base)
y_pred_wrapper = wrapper_model.predict(X_test_wrapper_scaled)

wrapper_rmse = np.sqrt(mean_squared_error(y_test_base, y_pred_wrapper))
wrapper_mae = mean_absolute_error(y_test_base, y_pred_wrapper)
wrapper_r2 = r2_score(y_test_base, y_pred_wrapper)

print("\n" + "-"*80)
print("WRAPPER METHOD MODEL PERFORMANCE")
print("-"*80)
print(f"Number of Features: {len(rfe_features)}")
print(f"RMSE: {wrapper_rmse:.4f}")
print(f"MAE:  {wrapper_mae:.4f}")
print(f"R²:   {wrapper_r2:.4f}")

# Store results
results['Method'].append('Wrapper Methods (RFE)')
results['Number_of_Features'].append(len(rfe_features))
results['RMSE'].append(wrapper_rmse)
results['MAE'].append(wrapper_mae)
results['R2'].append(wrapper_r2)

## Cell 15: Embedded Method - LASSO Regression

In [ ]:
print("\n" + "="*80)
print("EMBEDDED METHOD: LASSO REGRESSION")
print("="*80)

# LassoCV to find optimal alpha
lasso_cv = LassoCV(cv=5, random_state=42, max_iter=10000)
lasso_cv.fit(X_train_base_scaled, y_train_base)

optimal_alpha = lasso_cv.alpha_
print(f"\nOptimal Alpha (λ): {optimal_alpha:.6f}")

# Train final LASSO model
lasso_model = Lasso(alpha=optimal_alpha, max_iter=10000)
lasso_model.fit(X_train_base_scaled, y_train_base)

# Extract non-zero coefficients
lasso_coef = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': np.abs(lasso_model.coef_)
}).sort_values('Coefficient', ascending=False)

lasso_features = lasso_coef[lasso_coef['Coefficient'] > 0]['Feature'].tolist()

print(f"\nFeatures with non-zero coefficients: {len(lasso_features)}")
for i, feat in enumerate(lasso_features, 1):
    coef = lasso_model.coef_[X.columns.get_loc(feat)]
    print(f"{i:2d}. {feat:30s} (Coefficient: {coef:8.4f})")

print("\n" + "-"*80)
print("LASSO COEFFICIENTS (All Features)")
print("-"*80)
print(lasso_coef.to_string(index=False))

# Plot LASSO coefficients
plt.figure(figsize=(10, 6))
colors = ['darkred' if x == 0 else 'darkgreen' for x in lasso_coef['Coefficient']]
plt.barh(lasso_coef['Feature'], lasso_coef['Coefficient'], color=colors)
plt.xlabel('Absolute Coefficient Value', fontsize=12, fontweight='bold')
plt.ylabel('Features', fontsize=12, fontweight='bold')
plt.title('LASSO Coefficient Importance\n(Green: Selected, Red: Not Selected)', 
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Cell 16: Embedded Method Evaluation

In [ ]:
print("\n" + "="*80)
print("EMBEDDED METHOD EVALUATION")
print("="*80)

# If very few features selected, use top 5
if len(lasso_features) < 3:
    lasso_features = lasso_coef.head(5)['Feature'].tolist()
    print(f"Note: LASSO selected very few features. Using top 5 for evaluation.")

print(f"\nNumber of Features: {len(lasso_features)}")

# Prepare data
X_train_embedded = X_train_base[lasso_features]
X_test_embedded = X_test_base[lasso_features]

# Standardize
scaler_embedded = StandardScaler()
X_train_embedded_scaled = scaler_embedded.fit_transform(X_train_embedded)
X_test_embedded_scaled = scaler_embedded.transform(X_test_embedded)

# Train and evaluate
embedded_model = LinearRegression()
embedded_model.fit(X_train_embedded_scaled, y_train_base)
y_pred_embedded = embedded_model.predict(X_test_embedded_scaled)

embedded_rmse = np.sqrt(mean_squared_error(y_test_base, y_pred_embedded))
embedded_mae = mean_absolute_error(y_test_base, y_pred_embedded)
embedded_r2 = r2_score(y_test_base, y_pred_embedded)

print("\n" + "-"*80)
print("EMBEDDED METHOD MODEL PERFORMANCE")
print("-"*80)
print(f"Number of Features: {len(lasso_features)}")
print(f"RMSE: {embedded_rmse:.4f}")
print(f"MAE:  {embedded_mae:.4f}")
print(f"R²:   {embedded_r2:.4f}")

# Store results
results['Method'].append('Embedded Method (LASSO)')
results['Number_of_Features'].append(len(lasso_features))
results['RMSE'].append(embedded_rmse)
results['MAE'].append(embedded_mae)
results['R2'].append(embedded_r2)

## Cell 17: Comparison of All Methods

In [ ]:
print("\n" + "="*80)
print("COMPREHENSIVE COMPARISON OF ALL METHODS")
print("="*80)

# Create results dataframe
results_df = pd.DataFrame(results)
results_df_sorted = results_df.sort_values('RMSE')

print("\n" + "-"*80)
print("COMPARISON TABLE (Sorted by RMSE - Lower is Better)")
print("-"*80)
print(results_df_sorted.to_string(index=False))

# Find best method
best_method_idx = results_df_sorted['RMSE'].idxmin()
best_method = results_df_sorted.loc[best_method_idx]

print("\n" + "="*80)
print("🏆 BEST PERFORMING METHOD")
print("="*80)
print(f"Method: {best_method['Method']}")
print(f"Number of Features: {int(best_method['Number_of_Features'])}")
print(f"RMSE: {best_method['RMSE']:.4f}")
print(f"MAE:  {best_method['MAE']:.4f}")
print(f"R²:   {best_method['R2']:.4f}")

# Performance improvement
improvement_rmse = ((baseline_rmse - best_method['RMSE']) / baseline_rmse) * 100
print(f"\nRMSE Improvement vs Baseline: {improvement_rmse:+.2f}%")

## Cell 18: Performance Comparison Visualizations

In [ ]:
print("\nCreating performance comparison visualizations...")

# Create a figure with 4 subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# RMSE Comparison
ax1 = axes[0, 0]
colors_rmse = ['#d32f2f' if m == best_method['Method'] else '#1976d2' 
               for m in results_df_sorted['Method']]
ax1.barh(results_df_sorted['Method'], results_df_sorted['RMSE'], color=colors_rmse)
ax1.set_xlabel('RMSE (Lower is Better)', fontweight='bold')
ax1.set_title('RMSE Comparison', fontweight='bold')
ax1.invert_yaxis()

# MAE Comparison
ax2 = axes[0, 1]
colors_mae = ['#d32f2f' if m == best_method['Method'] else '#1976d2' 
              for m in results_df_sorted['Method']]
ax2.barh(results_df_sorted['Method'], results_df_sorted['MAE'], color=colors_mae)
ax2.set_xlabel('MAE (Lower is Better)', fontweight='bold')
ax2.set_title('MAE Comparison', fontweight='bold')
ax2.invert_yaxis()

# R² Comparison
ax3 = axes[1, 0]
colors_r2 = ['#4caf50' if m == best_method['Method'] else '#1976d2' 
             for m in results_df_sorted['Method']]
ax3.barh(results_df_sorted['Method'], results_df_sorted['R2'], color=colors_r2)
ax3.set_xlabel('R² Score (Higher is Better)', fontweight='bold')
ax3.set_title('R² Score Comparison', fontweight='bold')
ax3.invert_yaxis()

# Feature Count Comparison
ax4 = axes[1, 1]
colors_feat = ['#ff6f00' if m == best_method['Method'] else '#1976d2' 
               for m in results_df_sorted['Method']]
ax4.barh(results_df_sorted['Method'], results_df_sorted['Number_of_Features'], color=colors_feat)
ax4.set_xlabel('Number of Features', fontweight='bold')
ax4.set_title('Feature Count Comparison', fontweight='bold')
ax4.invert_yaxis()

plt.suptitle('Performance Comparison: All Feature Selection Methods', 
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("✓ Comparison visualizations created!")

## Cell 19: Final Analysis Report

In [ ]:
print("\n" + "="*80)
print("FINAL ANALYSIS AND CONCLUSIONS")
print("="*80)

analysis_report = f"""
{'='*80}
WINE QUALITY DATASET - FEATURE SELECTION ANALYSIS REPORT
{'='*80}

EXECUTIVE SUMMARY
{'-'*80}
This analysis evaluated three major feature selection approaches on the Wine 
Quality (white) dataset: Filter Methods, Wrapper Methods, and Embedded Methods.

DATASET OVERVIEW
{'-'*80}
- Total Samples: {df.shape[0]}
- Total Features: {df.shape[1] - 1}  (excluding target variable)
- Target Variable: Quality (regression task)
- Train-Test Split: 80/20

KEY FINDINGS
{'-'*80}

RESULTS COMPARISON (Sorted by RMSE)

{results_df_sorted.to_string(index=False)}

BEST PERFORMING METHOD: {best_method['Method']}
- Features: {int(best_method['Number_of_Features'])}
- RMSE: {best_method['RMSE']:.4f}
- MAE: {best_method['MAE']:.4f}
- R²: {best_method['R2']:.4f}
- Improvement: {improvement_rmse:+.2f}% vs Baseline

CONCLUSIONS
{'-'*80}

1. All feature selection methods successfully reduced features while maintaining
   or improving model performance.

2. The {best_method['Method']} achieved the best performance, selecting 
   {int(best_method['Number_of_Features'])} features from {df.shape[1]-1} original features.

3. Features consistently selected across methods indicate their importance for
   predicting wine quality.

4. Feature reduction improved model efficiency and interpretability.

RECOMMENDATIONS
{'-'*80}

For Initial Analysis: Use FILTER METHODS
- Fast and interpretable
- Good for exploratory analysis

For Best Performance: Use WRAPPER METHODS (RFE)
- Better accuracy
- Considers feature interactions

For Production Systems: Use EMBEDDED METHODS (LASSO)
- Efficient and automated
- Built-in regularization

Best Practice: Ensemble Approach
- Combine all three methods
- Select features appearing in multiple methods
- Leads to robust, generalizable models

{'='*80}
"""

print(analysis_report)

## Cell 20: Summary Statistics

In [ ]:
print("\n" + "="*80)
print("FINAL SUMMARY STATISTICS")
print("="*80)

summary_table = f"""
{'-'*80}
MODEL PERFORMANCE SUMMARY
{'-'*80}
{'Method':<30} {'Features':>8} {'RMSE':>10} {'MAE':>10} {'R²':>10}
{'-'*80}
"""

for _, row in results_df_sorted.iterrows():
    summary_table += f"{row['Method']:<30} {int(row['Number_of_Features']):>8} {row['RMSE']:>10.4f} {row['MAE']:>10.4f} {row['R2']:>10.4f}\n"

summary_table += f"{'-'*80}\n"

feature_stats = f"""
{'-'*80}
FEATURE SELECTION STATISTICS
{'-'*80}
Original Features:                    {df.shape[1] - 1}
Features Selected (Filter):           {len(filter_features)}
Features Selected (Wrapper/RFE):      {len(rfe_features)}
Features Selected (Embedded/LASSO):   {len(lasso_features)}
Average Features Selected:            {(len(filter_features) + len(rfe_features) + len(lasso_features)) / 3:.1f}
Feature Reduction Rate:               {(1 - (len(rfe_features) / (df.shape[1] - 1))) * 100:.1f}%
{'-'*80}
"""

print(summary_table)
print(feature_stats)

print("\n" + "="*80)
print("✓ FEATURE SELECTION ANALYSIS COMPLETE!")
print("="*80)